# Comparison of Text Classifiers: RNN, GRU, and LSTM

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import io
import re
from collections import Counter
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from torch.utils.data import DataLoader, Dataset

## dataset preprocessing

In [ ]:
vocab_size = 10000
max_length = 256
batch_size = 64

print("Please upload your 'IMDB Dataset.csv' file:")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[file_name]))
df['sentiment'] = df['sentiment'].map({'positive': 1.0, 'negative': 0.0})

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['review'].tolist(),
    df['sentiment'].tolist(),
    test_size=0.2,
    random_state=42
)

def tokenize(text):
    return re.findall(r'\b\w+\b', text.lower())

print("Building vocabulary...")
word_counts = Counter()
for text in train_texts:
    word_counts.update(tokenize(text))

most_common_words = word_counts.most_common(vocab_size - 2)

word_to_idx = {"<pad>": 0, "<unk>": 1}
for idx, (word, _) in enumerate(most_common_words, start=2):
    word_to_idx[word] = idx


def text_pipeline(text):
    tokens = tokenize(text)
    indices = [word_to_idx.get(word, word_to_idx["<unk>"]) for word in tokens]

    if len(indices) > max_length:
        return indices[:max_length]
    else:
        padding = [word_to_idx["<pad>"]] * (max_length - len(indices))
        return indices + padding

class IMDBCsvDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        seq = text_pipeline(text)

        return torch.tensor(seq, dtype=torch.long), torch.tensor(label, dtype=torch.float32)

print("Creating DataLoaders...")
train_dataset = IMDBCsvDataset(train_texts, train_labels)
test_dataset = IMDBCsvDataset(test_texts, test_labels)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Training samples: {len(train_dataset)} | Testing samples: {len(test_dataset)}")

Please upload your 'IMDB Dataset.csv' file:


Saving IMDB Dataset.csv to IMDB Dataset (1).csv
Building vocabulary...
Creating DataLoaders...
Training samples: 40000 | Testing samples: 10000


## Models architecture

In [ ]:
embed_dim = 100
hidden_dim = 128
num_layers = 1

class TextClassifier(nn.Module):
    def __init__(self, rnn_type, vocab_size, embed_dim, hidden_dim, num_layers):
        super(TextClassifier, self).__init__()
        self.rnn_type = rnn_type

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        if self.rnn_type == 'RNN':
            self.rnn = nn.RNN(embed_dim, hidden_dim, num_layers, batch_first=True)
        elif self.rnn_type == 'LSTM':
            self.rnn = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True)
        elif self.rnn_type == 'GRU':
            self.rnn = nn.GRU(embed_dim, hidden_dim, num_layers, batch_first=True)

        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.embedding(x)

        if self.rnn_type == 'LSTM':
            out, (hidden, cell) = self.rnn(embedded)
        else:
            out, hidden = self.rnn(embedded)

        last_out = out[:, -1, :]

        logits = self.fc(last_out)
        probs = self.sigmoid(logits)

        return probs.squeeze()


## training

In [ ]:
learning_rate = 0.001
epochs = 10

def train_model(model, dataloader, epochs, learning_rate):
    model.train()
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    for epoch in range(epochs):
        epoch_loss = 0
        for batch_inputs, batch_targets in dataloader:
            optimizer.zero_grad()

            outputs = model(batch_inputs)
            loss = criterion(outputs, batch_targets)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

            optimizer.step()
            epoch_loss += loss.item()

        print(f"{model.rnn_type} | Epoch {epoch+1}/{epochs} | Loss: {epoch_loss/len(dataloader):.4f}")

    return model

## Evaluation

In [ ]:
def evaluate_model(model, dataloader):
    model.eval()

    all_targets = []
    all_predictions = []

    with torch.no_grad():
        for batch_inputs, batch_targets in dataloader:
            probs = model(batch_inputs)
            preds = (probs >= 0.5).float()

            all_targets.extend(batch_targets.cpu().numpy())
            all_predictions.extend(preds.cpu().numpy())

    acc = accuracy_score(all_targets, all_predictions)
    prec = precision_score(all_targets, all_predictions)
    rec = recall_score(all_targets, all_predictions)
    f1 = f1_score(all_targets, all_predictions)
    cm = confusion_matrix(all_targets, all_predictions)

    print(f"\n--- {model.rnn_type} Metrics ---")
    print(f"Test Accuracy: {acc:.4f}")
    print(f"Precision:     {prec:.4f}")
    print(f"Recall:        {rec:.4f}")
    print(f"F1-Score:      {f1:.4f}")
    print(f"Confusion Matrix:\n{cm}")


whole pipeline, save the models

In [ ]:
model_types = ['RNN', 'LSTM', 'GRU']

for rnn_type in model_types:
    print(f"\nTraining {rnn_type}...")
    model = TextClassifier(rnn_type, vocab_size, embed_dim, hidden_dim, num_layers)

    trained_model = train_model(model, train_dataloader, epochs, learning_rate)
    evaluate_model(trained_model, test_dataloader)

    save_path = f"{rnn_type}_model.pth"
    torch.save(trained_model.state_dict(), save_path)
    print(f"--- Saved {rnn_type} model weights to '{save_path}' ---")


Training RNN...
RNN | Epoch 1/10 | Loss: 0.6972
RNN | Epoch 2/10 | Loss: 0.6917
RNN | Epoch 3/10 | Loss: 0.6884
RNN | Epoch 4/10 | Loss: 0.6766
RNN | Epoch 5/10 | Loss: 0.6619
RNN | Epoch 6/10 | Loss: 0.6407
RNN | Epoch 7/10 | Loss: 0.6152
RNN | Epoch 8/10 | Loss: 0.5886
RNN | Epoch 9/10 | Loss: 0.5661
RNN | Epoch 10/10 | Loss: 0.5448

--- RNN Metrics ---
Test Accuracy: 0.5142
Precision:     0.5556
Recall:        0.1794
F1-Score:      0.2712
Confusion Matrix:
[[4238  723]
 [4135  904]]
--- Saved RNN model weights to 'RNN_model.pth' ---

Training LSTM...
LSTM | Epoch 1/10 | Loss: 0.6930
LSTM | Epoch 2/10 | Loss: 0.6841
LSTM | Epoch 3/10 | Loss: 0.6696
LSTM | Epoch 4/10 | Loss: 0.4860
LSTM | Epoch 5/10 | Loss: 0.2974
LSTM | Epoch 6/10 | Loss: 0.2285
LSTM | Epoch 7/10 | Loss: 0.1765
LSTM | Epoch 8/10 | Loss: 0.1315
LSTM | Epoch 9/10 | Loss: 0.0970
LSTM | Epoch 10/10 | Loss: 0.0719

--- LSTM Metrics ---
Test Accuracy: 0.8774
Precision:     0.8831
Recall:        0.8722
F1-Score:      0.877

## fast demo

In [ ]:
def predict_sentiment(model, text, max_length=256):
    """Predicts the sentiment of a single custom text string."""
    model.eval()
    seq = text_pipeline(text)
    tensor_input = torch.tensor([seq], dtype=torch.long)
    with torch.no_grad():
        probability = model(tensor_input).item()
    sentiment = "Positive" if probability >= 0.5 else "Negative"

    return sentiment, probability


In [ ]:
custom_review = "The movie started off a bit slow, but the acting was absolutely phenomenal and the ending left me speechless. Highly recommend!"

print(f"Review: '{custom_review}'\n")

for target_model_type in ("RNN", "LSTM", "GRU"):

    # Re-initialize a new model of that type
    demo_model = TextClassifier(target_model_type, vocab_size, embed_dim, hidden_dim, num_layers)

    # Load the saved weights into the blank model
    load_path = f"{target_model_type}_model.pth"
    demo_model.load_state_dict(torch.load(load_path))

    # Run the prediction
    sentiment, prob = predict_sentiment(demo_model, custom_review, max_length)

    print(f"Model: {demo_model.rnn_type}")
    print(f"Predicted Sentiment: {sentiment}")
    print(f"Probability Score: {prob:.4f}")

Review: 'The movie started off a bit slow, but the acting was absolutely phenomenal and the ending left me speechless. Highly recommend!'

Model: RNN
Predicted Sentiment: Negative
Probability Score: 0.4966
Model: LSTM
Predicted Sentiment: Positive
Probability Score: 0.9975
Model: GRU
Predicted Sentiment: Positive
Probability Score: 0.9968
